# Exploratory Data Analysis (EDA) - ISCX Tabular Engine

This notebook serves as the **Research & Validation Phase** for the Tabular Structural Features engine. We explore the ISCX 2016 URL Dataset to understand class distributions, missing values, and feature bounds before feeding them into our 9 baseline models.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.style.use('ggplot')
sns.set_palette('muted')

### 1. Data Ingestion & Target Serialization
The dataset contains a multi-class string column (`URL_Type_obf_Type`). We must map this to a Binary distribution (`1` for Phishing/Malware/Defacement, `0` for Benign).

In [ ]:
df = pd.read_csv('../data/iscx_phishing/All.csv', low_memory=False)
print(f"Original ISCX Shape: {df.shape}")

target_col = 'URL_Type_obf_Type'
df['target'] = df[target_col].apply(lambda x: 0 if str(x).strip().lower() == 'benign' else 1)

plt.figure(figsize=(6, 4))
ax = sns.countplot(data=df, x='target')
plt.title('Binary Target Distribution (0 = Benign, 1 = Phishing/Malicious)')
plt.show()

### 2. Feature Filtering & Infinities
The ISCX dataset is prone to divide-by-zero errors resulting in `np.inf`. We must cast these to `NaN` so the `SimpleImputer` can safely transform them to medians.

In [ ]:
X = df.drop(columns=[target_col, 'target'], errors='ignore')
X = X.select_dtypes(include=[np.number])

# Check for Infinities
inf_count = np.isinf(X).values.sum()
print(f"Total Infinity Values Detected: {inf_count}")

# Scrub infinities
X = X.replace([np.inf, -np.inf], np.nan)
print(f"Infinities scrubbed. Matrix ready for SimpleImputer pipeline.")